# Evaluating The Base Model for Guessing Item Prices

In [41]:
# Imports

import os

from accelerate.inference import generate_device_map
from dotenv import load_dotenv
from tqdm import tqdm
from huggingface_hub import login
from datasets import load_dataset
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from pricer.util import evaluate
import matplotlib.pyplot as plt

In [2]:
# Constants:
BASE_MODEL = 'meta-llama/Llama-3.2-3B'

In [6]:
# Log in to Hugging Face:
load_dotenv(override= True)
hf_token = os.getenv('HF_TOKEN')
login(token= hf_token, add_to_git_credential= True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Loading in Dataset:

In [10]:
# Loading in Dataset:
dataset_name = 'Shailya01/items_prompt_lite'
dataset = load_dataset(path= dataset_name)
train = dataset['train']
val = dataset['val']
test = dataset['test']

print(f'Loaded {len(train)} training items, {len(val)} validation items, {len(test)} test items')

Loaded 20000 training items, 1000 validation items, 1000 test items


In [11]:
train[0]

{'prompt': 'What does this cost to the nearest dollar?\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.\n\nPrice is $',
 'completion': '64.00'}

In [12]:
val[0]

{'prompt': 'What does this cost to the nearest dollar?\n\nTitle: SAFUEL 10,000mAh Magnetic Portable Charger  \nCategory: Electronics  \nBrand: SAFUEL  \nDescription: A compact 10,000mAh power bank that snaps magnetically to MagSafe‑enabled iPhones for fast wireless charging.  \nDetails: Offers 20W USB‑C PD, QC 3.0 wired output, auto shut‑off, and a built‑in LED indicator.\n\nPrice is $',
 'completion': '33.00'}

In [13]:
test[0]

{'prompt': 'What does this cost to the nearest dollar?\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.\n\nPrice is $',
 'completion': '219.0'}

## Loading in Quantized Base Mdoel:

In [14]:
# Quantization Configuration:
quant_config = BitsAndBytesConfig(
    load_in_4bit= True,
    bnb_4bit_compute_dtype= torch.bfloat16,
    bnb_4bit_use_double_quant= True,
    bnb_4bit_quant_type= 'nf4'
)

In [17]:
# Loading in Tokenizer and Model:
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=BASE_MODEL, trust_remote_code= True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
base_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path= BASE_MODEL,
    quantization_config= quant_config,
    device_map= 'auto'
)

print(f'Memory Footprint of Base Model: {base_model.get_memory_footprint() / 1e9:.2f}GB')

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Memory Footprint of Base Model: 2.20GB


## Model Prediction Function:

In [44]:
def model_predict(item):
    inputs = tokenizer(item['prompt'], return_tensors='pt').to('cuda')

    with torch.no_grad():
        output_ids = base_model.generate(**inputs, max_new_tokens= 5, pad_token_id= tokenizer.eos_token_id)

    prompt_len = inputs['input_ids'].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

In [45]:
test[0]

{'prompt': 'What does this cost to the nearest dollar?\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.\n\nPrice is $',
 'completion': '219.0'}

In [46]:
model_predict(test[0])

'149.99, and'